In [1]:
import modules.data as d
import modules.model as m
import modules.train as t
import modules.utils as u

import torch
import torch.nn as nn
from pathlib import Path

---

In [2]:
# dataset dir
datasets = Path('/home/mv18gs/Documents/GitHub/pathway_model/datasets/')

# get device
device, generator = u.Devices().auto_set_device()

# get data
data = d.Data(
    # datasets
    tcga_project = 'TCGA-BRCA',
    metadata_subtype_col = 'paper_BRCA_Subtype_PAM50',

    # dirs
    tcga_dir = datasets / 'tcga',
    relation_filepath = datasets / 'other/relation_ohe.csv',
    
    # col, filter
    y_col = 'subtype',
    drop = {'subtype':['Normal', 'Metastatic']},
    max_subset=120,
)

*** Device() ***
device = cuda:3

**** Data() ****
log0_method      log1p            str
class_weights    (6,)             Tensor (cuda:3)
gene_counts      (4383, 567)      DataFrame
metadata         (567, 3)         DataFrame
relation         (32798, 18)      DataFrame
node_id_map      4383             dict
masks            305              list
X                (567, 4383, 1)   Tensor (cuda:3)
y                (567, 6)         Tensor (cuda:3)
y_labels         6                list
num_samples      567              int
num_nodes        4383             int
num_features     1                int
num_labels       6                int
num_masks        305              int



---

In [5]:
import sklearn.metrics
import numpy as np
from modules.train import Trainer
from torch import Tensor

In [6]:
class ClassificationTrainer(Trainer):
    def _compute_metrics(self, batch):
        # init
        metrics = {}

        # get predictions
        out = torch.Tensor(np.array(batch['out']))
        y_pred = self.model.predict(out).detach().cpu().numpy()

        # compute metrics
        metrics['loss'] = batch['loss']/batch['len']
        metrics['accuracy'] = sklearn.metrics.accuracy_score(y_true=batch['y'], y_pred=y_pred)

        kwargs = {
            'y_true': batch['y'],
            'y_pred': y_pred,
            'average': 'weighted',
            'zero_division': 0
        }

        metrics['precision'] = sklearn.metrics.precision_score(**kwargs)
        metrics['recall'] = sklearn.metrics.recall_score(**kwargs)
        metrics['f1'] = sklearn.metrics.f1_score(**kwargs)
        
        return metrics

In [7]:
# get configs
model_config = t.Config(name='model')
model_config.add(
    key='classifier',
    model_class=m.MLPClassifier,
    model_kwargs={
        'in_features':data.num_nodes,
        'out_features':data.num_labels,
        'flatten':True,
    },
    trainer_class=ClassificationTrainer,
    trainer_kwargs={
        'loss_fn':nn.CrossEntropyLoss(data.class_weights),
        'verbose':False,
    }
)

# define experiment, run experiment
exp = t.Experiment(
    num_trials=3,
    num_epochs=10,
    X=data.X,
    y=data.y,
    generator=generator,
)
exp.add_config(model_config)
exp.run_experiment(comment='trainer_test', verbose=True)

100%|██████████| 3/3 [00:02<00:00,  1.50it/s]


---

In [18]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [27]:
class RegressionTrainer(Trainer):
    def _compute_metrics(self, batch):
        # init
        metrics = {}

        # concatenate arrays
        out = np.concatenate([i.flatten() for i in batch['out']])
        y = np.concatenate([i.flatten() for i in batch['y']])

        # calcualte metrics
        kwargs={'y_true':y, 'y_pred':out}
        metrics['mse'] = mean_squared_error(**kwargs)
        metrics['rmse'] = np.sqrt(metrics['mse'])
        metrics['mae'] = mean_absolute_error(**kwargs)
        metrics['corr'] = np.corrcoef(out, y)[0,1]

        return metrics

In [28]:
# get configs
model_config = t.Config(name='model')
model_config.add(
    key='classifier',
    model_class=m.MLPAutoencoder,
    model_kwargs={
        'in_features':data.num_nodes,
        'embedding_size':8,
        'squeeze':True,
        'unsqueeze':True
    },
    trainer_class=RegressionTrainer,
    trainer_kwargs={
        'loss_fn':nn.L1Loss(),
        'verbose':False,
    }
)

# define experiment, run experiment
exp = t.Experiment(
    num_trials=3,
    num_epochs=10,
    X=data.X,
    y=data.X,
    generator=generator,
)
exp.add_config(model_config)
exp.run_experiment(comment='trainer_test', verbose=True)

100%|██████████| 3/3 [00:03<00:00,  1.10s/it]


---

In [42]:
# model_config.trainer['classifier'].model

MLPAutoencoder(
  (encoder): MLP(
    (model): Sequential(
      (0): Linear(in_features=4383, out_features=8, bias=False)
    )
  )
  (decoder): MLP(
    (model): Sequential(
      (0): Linear(in_features=8, out_features=4383, bias=False)
    )
  )
)

In [ ]:
# model_config.trainer['classifier'].loader.

In [ ]:
# from modules.train import Config, Loader
# from torch.utils.data import TensorDataset, DataLoader

In [ ]:
# class AEPredictorConfig(Config):
#     def _pipeline(self, loader:Loader, ae_key, classifier_key, batch_size:int):
#         # train ae
#         self.trainer[ae_key].run()

#         # get ae model
#         ae = self.trainer[ae_key].model

#         # extract embeddings
#         ae.eval()

#         def _get_embeddings(dataloader):
#             embeddings = []
#             ys = []
            
#             with torch.no_grad():
#                 for X, y in dataloader:
#                     embedding = ae.encode(data)
#                     embeddings.append(embedding)
#                     ys.append(y)

#             embeddings_dataset = torch.cat(embeddings)
#             ys_dataset = torch.cat(ys)

#             embedding_ds = TensorDataset(embeddings_dataset, ys_dataset)
#             embedding_loader = DataLoader(embedding_ds, batch_size=batch_size, shuffle=True)
        